# Preparing ABR Imagery 2021 Data for Analysis

**Will be focusing on SO Detail History View**

In [1]:
#import packages
import pandas as pd
import numpy as np
from datetime import datetime
from os import devnull
import xlrd

In [2]:
#since SO Detail History View is in .Xls format, will need to use devnull and xlrd to open
wb = xlrd.open_workbook('SO Detail History View 2021.Xls', logfile = open(devnull, 'w'))
data = pd.read_excel(wb, thousands = ',')

## **Filtering to keep orders MADE in 2021**

In [3]:
#making transaction date and Actu Ship Date  datetime objects
data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], format = "%m/%d/%Y")
data['Act Ship Date'] = pd.to_datetime(data['Act Ship Date'], format = "%m/%d/%Y")

In [4]:
#filtering data made in 2021
data= data[(data['Transaction Date'] >= '2021-01-01') & (data['Transaction Date'] <= '2021-12-31')]

In [5]:
#testing to see that it worked, great it does!
print(data['Transaction Date'].min())
print(data['Transaction Date'].max())

2021-01-01 00:00:00
2021-12-01 00:00:00


# Changing type of Transaction ID to string and formating to match Traverse

In [6]:
data['Transaction ID'] = data['Transaction ID'].astype({'Transaction ID' : 'float'})
data['Transaction ID'] = data['Transaction ID'].astype({'Transaction ID' : 'int'})
data['Transaction ID'] = data['Transaction ID'].astype({'Transaction ID' : str})

In [7]:
data['Transaction ID'] = data['Transaction ID'].apply(lambda x: x.zfill(8))

In [8]:
data.head()
#sucessful

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Item ID,Location ID,Price Code,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-100.79
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-620.62
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-74.09
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-400.04
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-128.90


## Filtering out Customer ID's that are not actually customers

In [9]:
customer_id_filter = ['edr0001','plt0001','Jos0118','Mag0003', 'PDK0001', 'isg0001']
data = data[~data['Customer ID'].isin(customer_id_filter)]

In [10]:
data.head()

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Item ID,Location ID,Price Code,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-100.79
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-620.62
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-74.09
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-400.04
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-128.90


## Saving unique items to text file to view them for any items that should't be there

In [11]:
np.savetxt('unique items 2021.txt', data['Item ID'].unique(), fmt='%s')

**All the items look good**

# There are many items with no Price Code, there is another alternative to use called Product Line. Will add this to the table replacing Price Code. Will check Product Line's and make sure all items have Product Lines

**In Traverse, got a list of all the items called 'IN Item View ALL.Xls'. This has the column called Product Line that we want. I will load this in and join with data on 'Item ID'**

In [12]:
#loading items
wb = xlrd.open_workbook('IN Item View ALL.Xls', logfile = open(devnull, 'w'))
items = pd.read_excel(wb)

In [13]:
#filtering to have only item id and prodcut lin
items = items[['Item ID', 'Product Line', 'Additional Description']]

In [14]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15646 entries, 0 to 15645
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Item ID                 15646 non-null  object
 1   Product Line            15628 non-null  object
 2   Additional Description  15467 non-null  object
dtypes: object(3)
memory usage: 366.8+ KB


In [15]:
#if there are any duplicated Items, they must be removed, this is so the join will not duplicated item ids for the orders
items.loc[(items['Item ID'].duplicated())]

,Item ID,Product Line,Additional Description
31,0041012,MORETTI TRAN,This crystal-clear Moretti soft glass rod is 1...
33,0041415,MORETTI TRAN,This crystal-clear Moretti soft glass rod is 1...
35,00456,MORETTI TRAN,This crystal-clear Moretti soft glass rod is 5...
355,1556008,MORETTI TRAN,Amber Pale Moretti Color Rod 104coe. Availab...
357,1556012,MORETTI TRAN,Amber Light Moretti Color Rod 104coe. Availa...
...,...,...,...
15617,YTC,Asian Colore,This transparent dark amber yellow tubing is m...
15624,YTC32,Asian Colore,This transparent dark amber yellow tubing is m...
15626,YTC38,Asian Colore,This transparent dark amber yellow tubing is m...
15629,YTC44,Asian Colore,This transparent dark amber yellow tubing is m...


In [16]:
items.drop_duplicates(subset = 'Item ID', inplace = True)

In [17]:
#joining items with data
data = data.merge(items, on = 'Item ID', how = 'left')

In [18]:
data[data['Transaction ID'] == '00178603']

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Item ID,Location ID,Price Code,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount,Product Line,Additional Description
20865,Line Item,00178603,jas0097,2021-06-15,2021-06-16,Invoice,Asian 25.4mm x 4.0mm Clear Tubing,AT2540,B01,ASIAN GLAS,1.0,CS,124.0,0.0,Percent,6.634,NaN,124.0,ASIAN TUBE,This economically priced borosilicate glass tu...
20866,Line Item,00178603,jas0097,2021-06-15,2021-06-16,Invoice,Asian 16mmx2.5mm-3.2mm Clear Tubing,AT1632,B01,ASIAN GLAS,1.0,CS,80.0,0.0,Percent,15.743,NaN,80.0,ASIAN TUBE,This economically priced borosilicate glass tu...
20867,Freight,00178603,jas0097,2021-06-15,2021-06-16,Invoice,NaN,NaN,B01,NaN,1.0,NaN,125.0,0.0,Amount,0.000,NaN,125.0,NaN,NaN


In [19]:
#removing Price code
del data['Price Code']



In [20]:
#reordering dataframe
data = data[['History Type','Transaction ID','Customer ID','Transaction Date','Act Ship Date','Transaction Type', 'Description','Additional Description','Item ID',
             'Location ID', 'Product Line','Qty Ship Sell', 'Units ','Unit Price Sell', 'Price Adj Pct', 'Price Adj Type', 'Unit Cost Sell', 'Ship Via', 'Amount']]

In [21]:
#renaming units to remove whitespace
data = data.rename(columns = {'Units ': 'Units'})

In [22]:
data.head()

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-100.79
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-620.62
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-74.09
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-400.04
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,Amount,0.0,NaN,-128.90


In [23]:
#viewing unique items that have no Product Line
data['Item ID'][(data['History Type'] == 'Line Item') & (data['Product Line'].isna())].unique()   
#why didn't this give me ALL the product lines with nan and history type line item ?

array(['purr-minor kit', 'purr-bravo kit', nan, 'claw11',
       'jc-3m2fd-ac-4.5', '854-203DSK-ADPT', 'Liftgatefee', 'JC-ST14',
       'JC-ST19', 'purr-glass pack', 'BB-46-B-BROWNHONEY', 'sun',
       'ind-3mc-b2', 'PORTABLECCASE2', '0000-BOROBATCH-LOGO-2XL',
       'crucible30lb', '4-2-INLET-CONVERSIONKIT',
       'Crucible-3inchby1.5inch', 'BEGINNERKIT-NORTELREDMAX',
       'Purr-Bravo Kit', 'GORO3D', 'GORO5', 'GOS6W', 'VW1-01680',
       'CLAW15', 'TUNGSTEN PICK 1/8', 'JC-1014B', 'OPAL POLISHED WHITE',
       'BB-03-M-DAVESBLUES', '854-220k', 'ind-14/10f-hw', 'ind-1fc-b2'],
      dtype=object)

### Adding Product Lines to Items with NaN Product Lines

**I have looked in Traverse for these Item ID's and have found their corresponding Product Line's. The only items that needed made Product Lines were crucible pots**

In [24]:
#replacing Prodcut lines
data.loc[(data['Item ID'] == 'LIftgatefee'), 'Product Line'] = 'service'
data.loc[(data['Item ID'] == 'liftgatefee'), 'Product Line'] = 'service'
data.loc[(data['Item ID'] == 'Liftgatefee'), 'Product Line'] = 'service'

purr_list = ['purr-minor kit', 'purr-bravo kit', 'purr-glass pack', 'Purr-Bravo Kit']
data.loc[(data['Item ID'].isin(purr_list)), 'Product Line'] = 'PURR'

hose_list = ['4-2-INLET-CONVERSIONKIT', '854-203DSK-ADPT', '854-220k']
data.loc[(data['Item ID'].isin(hose_list)), 'Product Line'] = 'Hoses & Acce'

bb_list = ['BB-03-M-DAVESBLUES', 'BB-46-B-BROWNHONEY']
data.loc[(data['Item ID'].isin(bb_list)), 'Product Line'] = 'Boro Batch'

data.loc[(data['Item ID'] == 'BEGINNERKIT-NORTELREDMAX'), 'Product Line'] = 'BEGINNER KIT'

claw_list = ['claw11', 'CLAW15']
data.loc[(data['Item ID'].isin(claw_list)), 'Product Line'] = 'CLAWS'

crucible_list = ['crucible30lb', 'Crucible-3inchby1.5inch']
data.loc[(data['Item ID'].isin(crucible_list)), 'Product Line'] = 'CRUCIBLE'

opal_list = ['GORO3D', 'GORO5', 'GOS6W', 'OPAL POLISHED WHITE']
data.loc[(data['Item ID'].isin(opal_list)), 'Product Line'] = 'OPALS'

joints_list = ['ind-14/10f-hw', 'ind-1fc-b2', 'ind-3mc-b2', 'JC-ST14', 'JC-ST19']
data.loc[(data['Item ID'].isin(joints_list)), 'Product Line'] = 'Glass Joints'

bushings_list = ['JC-1014B', 'jc-3m2fd-ac-4.5']
data.loc[(data['Item ID'].isin(bushings_list)), 'Product Line'] = 'Bushings'

data.loc[(data['Item ID'].isin(['PORTABLECCASE2'])), 'Product Line'] = 'CRAFT ITEMS'

data.loc[(data['Item ID'].isin(['TUNGSTEN PICK 1/8'])), 'Product Line'] = 'RAKES/PICKS'

data.loc[(data['Item ID'].isin(['VW1-01680'])), 'Product Line'] = 'HA TOOLS'

data.loc[(data['Item ID'].isin(['0000-BOROBATCH-LOGO-2XL'])), 'Product Line'] = 'T-SHIRTS'
#dropping the row with 'sun' as the item id
data = data.drop(data.loc[data['Item ID'] == 'sun'].index)



In [25]:
#further exploring line items with nan product line
data.loc[(data['Product Line'].isna()), 'History Type'].value_counts()

Payment      5543
Freight      4482
Sales Tax     522
Line Item      22
Misc            1
Name: History Type, dtype: int64

In [26]:
data.loc[(data['Product Line'].isna()) & (data['History Type'] == 'Line Item')]

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
649,Line Item,00181515,tom0001,2021-11-26,2021-11-30,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,GND,0.0
3429,Line Item,00181145,ste0474,2021-11-05,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
5371,Line Item,00180835,ian0020,2021-10-20,2021-10-20,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
8644,Line Item,00180338,dou0045,2021-09-23,2021-09-24,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
9294,Line Item,00180247,pau0049,2021-09-17,2021-09-20,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
12127,Line Item,00179869,pau0081,2021-08-26,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
13927,Line Item,00179578,Nor0008,2021-08-12,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
20499,Line Item,00178647,UOP0001,2021-06-17,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,NaN,0.0
20717,Line Item,00178609,ala0056,2021-06-16,NaT,Invoice,NaN,NaN,NaN,B01,NaN,0.0,LB,0.00,0.0,Percent,0.0,GND,0.0
21574,Line Item,00178439,mar1061,2021-06-09,2021-06-11,Invoice,NaN,NaN,NaN,B01,NaN,1.0,LB,0.00,0.0,Percent,0.0,NaN,0.0


**Looking at the cell above, everything with practically all NaN is empty rows,  I know this by double checking the orders in Traverse, These rows can be removed.
can remove Trans ID 00178332
can remove index 41715, this is an emtpy line with information on items that were subbed out
so everything from the table above can be removed from the dataset**


In [27]:
remove_index = data.loc[(data['Product Line'].isna()) & (data['History Type'] == 'Line Item')].index

data = data[~data.index.isin(remove_index)]

In [28]:
#no all the line items are dealt with
data.loc[(data['Product Line'].isna()), 'History Type'].value_counts()

Payment      5543
Freight      4482
Sales Tax     522
Misc            1
Name: History Type, dtype: int64

In [29]:
#looking at the miscallaneuous item in Traverse, this item needs to stay as it counts to what the customer paid
data.loc[(data['History Type'] == 'Misc')]

#need to keep the Freight and Sales Tax rows as well, this gives important information and these should not have a product line. in the future on this project, I may
# add product lines for these for help on getting totals for these categories, but for now, I will not put a Product Line on these

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
10088,Misc,00180135,sam0076,2021-09-10,2021-09-10,Invoice,NaN,NaN,NaN,B01,NaN,1.0,NaN,645.0,0.0,Amount,0.0,NaN,645.0


In [30]:
#changing dtypes of the data for values that should be numeric
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 47682 entries, 0 to 47704
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   History Type            47682 non-null  object        
 1   Transaction ID          47682 non-null  object        
 2   Customer ID             47682 non-null  object        
 3   Transaction Date        47682 non-null  datetime64[ns]
 4   Act Ship Date           40834 non-null  datetime64[ns]
 5   Transaction Type        47682 non-null  object        
 6   Description             37079 non-null  object        
 7   Additional Description  27281 non-null  object        
 8   Item ID                 37134 non-null  object        
 9   Location ID             42139 non-null  object        
 10  Product Line            37134 non-null  object        
 11  Qty Ship Sell           47682 non-null  float64       
 12  Units                   37134 non-null  object

In [31]:
data['Qty Ship Sell'] = data['Qty Ship Sell'].astype({'Qty Ship Sell' : float})
data['Unit Price Sell'] = data['Unit Price Sell'].astype({'Unit Price Sell' : float})
data['Amount'] = data['Amount'].astype({'Amount' : float})

# Adding a Drop Ship column to the data.

**Some items are drop ship items, will add a column detailing whether a particular item is a drop ship item or not.  An item will be a drop ship item if "DROP SHIP" exists in the Additional Description**

In [32]:
#setting nan in additional Description to 'No Additional Description'
data['Additional Description'] = data['Additional Description'].fillna('No Additional Description')

In [33]:
data['Additional Description'].str.contains(pat = 'DROP SHIP')

0        False
1        False
2        False
3        False
4        False
         ...  
47700    False
47701    False
47702    False
47703    False
47704    False
Name: Additional Description, Length: 47682, dtype: bool

In [34]:
#adding column for whether an item is Drop ship 
data['Drop Ship'] = 0

In [35]:
#creatiing conditional for Drop ship column, will either be Y or N
data.loc[(data['Additional Description'].str.contains(pat = 'DROP SHIP') == True), 'Drop Ship'] = 'Y'
data.loc[(data['Additional Description'].str.contains(pat = 'DROP SHIP') == False), 'Drop Ship'] = 'N'

# Handling the rest of the NaN data in the dataset

In [36]:
data.isna().sum()

History Type                  0
Transaction ID                0
Customer ID                   0
Transaction Date              0
Act Ship Date              6848
Transaction Type              0
Description               10603
Additional Description        0
Item ID                   10548
Location ID                5543
Product Line              10548
Qty Ship Sell                 0
Units                     10548
Unit Price Sell               0
Price Adj Pct                 0
Price Adj Type                0
Unit Cost Sell                0
Ship Via                  14471
Amount                        0
Drop Ship                     0
dtype: int64

**for items that have NaT for actual shipping date, these are items that did not ship, will keep this NaT**

**Item ID, Product Line, and Units are all columns that do not have items, I will change these to 'No Item ID', 'No Product Line', 'No Units'**

In [37]:
data['Item ID'].fillna('No Item ID', inplace = True)
data['Product Line'].fillna('No Product Line', inplace = True)
data['Units'].fillna('No Units', inplace = True)

In [38]:
#for Ship via, the NaN values are unknown, so I will put 'unknown', this is because I would have to mainly go through over 10,000 entries to see what the shipping was
# in Traverse. And that is ridiculous
data.loc[(data['Ship Via'].isna())]

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount,Drop Ship
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-100.79,N
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-620.62,N
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-74.09,N
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-400.04,N
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-128.90,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47682,Line Item,00174470,die0005,2021-01-02,2021-01-11,Invoice,Northstar Galaxy Rod,<h4> Item name: </h4>\n\nNorthstar Galaxy Rod ...,ns152,B01,Northstar Ro,2.0,LB,58.5,0.0,Amount,32.5,NaN,117.00,N
47683,Freight,00174470,die0005,2021-01-02,2021-01-11,Invoice,NaN,No Additional Description,No Item ID,B01,No Product Line,1.0,No Units,240.0,0.0,Amount,0.0,NaN,240.00,N
47702,Payment,00191070,ran0051,2021-01-02,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-1592.50,N
47703,Payment,00191077,nic0216,2021-01-01,NaT,Payment,NaN,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,NaN,-1076.35,N


In [39]:
#replacing ship via nan with unknown
data['Ship Via'].fillna('Unknown', inplace = True)

In [40]:
#Replacing Description NaN with 'No Description'
data['Description'].fillna('No Description', inplace = True)

In [41]:
#Great, that handles all the NaN data!
data.isna().sum()

History Type                 0
Transaction ID               0
Customer ID                  0
Transaction Date             0
Act Ship Date             6848
Transaction Type             0
Description                  0
Additional Description       0
Item ID                      0
Location ID               5543
Product Line                 0
Qty Ship Sell                0
Units                        0
Unit Price Sell              0
Price Adj Pct                0
Price Adj Type               0
Unit Cost Sell               0
Ship Via                     0
Amount                       0
Drop Ship                    0
dtype: int64

In [42]:
data

,History Type,Transaction ID,Customer ID,Transaction Date,Act Ship Date,Transaction Type,Description,Additional Description,Item ID,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount,Drop Ship
0,Payment,00205712,bet0095,2021-12-01,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-100.79,N
1,Payment,00204681,Tod0111,2021-12-01,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-620.62,N
2,Payment,00205719,chr0867,2021-12-01,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-74.09,N
3,Payment,00205734,col0265,2021-12-01,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-400.04,N
4,Payment,00205720,jen0078,2021-12-01,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-128.90,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47700,Line Item,00174473,ada0153,2021-01-02,2021-01-11,Invoice,Y-Splitter w/ Valves (Propane),Propane Y with Valves on Each Side: for use...,YPV,B01,Hoses & Acce,1.0,EA,39.95,0.0,Amount,20.908,GND,39.95,N
47701,Freight,00174473,ada0153,2021-01-02,2021-01-11,Invoice,No Description,No Additional Description,No Item ID,B01,No Product Line,1.0,No Units,26.45,0.0,Amount,0.000,GND,26.45,N
47702,Payment,00191070,ran0051,2021-01-02,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-1592.50,N
47703,Payment,00191077,nic0216,2021-01-01,NaT,Payment,No Description,No Additional Description,No Item ID,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,0.000,Unknown,-1076.35,N


In [43]:
#restructuring data
data = data[['History Type','Transaction ID','Transaction Type','Customer ID','Transaction Date','Act Ship Date', 'Description','Additional Description','Item ID', 'Drop Ship',
             'Location ID', 'Product Line','Qty Ship Sell', 'Units','Unit Price Sell', 'Price Adj Pct', 'Price Adj Type', 'Unit Cost Sell', 'Ship Via', 'Amount']]

In [44]:
#resetting index
data.reset_index(drop = True, inplace = True)

# Adding geographic information to orders

**I have made a file from Traverse with the Shipping Information for the Orders in this dataset, the file is 'AR Transaction View ALL.Xls'. I will load this data in and join to the main dataset on Transaction ID**

In [45]:
#importing Customer address
wb = xlrd.open_workbook('AR Transaction History View ALL.Xls', logfile = open(devnull, 'w'))
customer_data = pd.read_excel(wb)

In [46]:
customer_data.head()

,Customer ID,Transaction ID,Transaction Type,Invoice Number,Ship To City,Ship To Region,Ship To Country,Invoice Date,Total
0,djm0002,181679,Invoice,120588,Bloomington,NaN,USA,12/1/2021,38.24
1,sas0013,181674,Invoice,120568,Los Angeles,CA,USA,12/1/2021,55.00
2,rya0020,181673,Invoice,120566,Bloomington,IN,USA,12/1/2021,378.45
3,sam0161,181672,Invoice,120570,Hammond,IN,USA,12/1/2021,169.06
4,pur0003,181671,Invoice,120575,Hemet,CA,USA,12/1/2021,600.00


In [47]:
#changing Transaction ID to fit Traverse format
customer_data['Transaction ID'] = customer_data['Transaction ID'].astype({'Transaction ID' : 'float'})
customer_data['Transaction ID'] = customer_data['Transaction ID'].astype({'Transaction ID' : 'int'})
customer_data['Transaction ID'] = customer_data['Transaction ID'].astype({'Transaction ID' : str})
customer_data['Transaction ID'] = customer_data['Transaction ID'].apply(lambda x: x.zfill(8))

In [48]:
#only need Customer ID, city, region and country
customer_data = customer_data[['Transaction ID', 'Ship To City', 'Ship To Region', 'Ship To Country']]

In [49]:
data.columns

Index(['History Type', 'Transaction ID', 'Transaction Type', 'Customer ID',
       'Transaction Date', 'Act Ship Date', 'Description',
       'Additional Description', 'Item ID', 'Drop Ship', 'Location ID',
       'Product Line', 'Qty Ship Sell', 'Units', 'Unit Price Sell',
       'Price Adj Pct', 'Price Adj Type', 'Unit Cost Sell', 'Ship Via',
       'Amount'],
      dtype='object')

In [50]:
#joining customer data with data
data = data.merge(customer_data, on = 'Transaction ID', how = 'left')

In [51]:
data = data[['History Type','Transaction ID', 'Transaction Type', 'Customer ID', 'Ship To City', 'Ship To Region', 'Ship To Country', 'Transaction Date','Act Ship Date',
             'Description', 'Additional Description','Item ID', 'Drop Ship','Location ID', 'Product Line','Qty Ship Sell', 'Units','Unit Price Sell', 'Price Adj Pct', 
             'Price Adj Type', 'Unit Cost Sell', 'Ship Via', 'Amount']]

In [52]:
data.head()

,History Type,Transaction ID,Transaction Type,Customer ID,Ship To City,Ship To Region,Ship To Country,Transaction Date,Act Ship Date,Description,...,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Unit Cost Sell,Ship Via,Amount
0,Payment,00205712,Payment,bet0095,NaN,NaN,NaN,2021-12-01,NaT,No Description,...,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,Unknown,-100.79
1,Payment,00204681,Payment,Tod0111,NaN,NaN,NaN,2021-12-01,NaT,No Description,...,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,Unknown,-620.62
2,Payment,00205719,Payment,chr0867,NaN,NaN,NaN,2021-12-01,NaT,No Description,...,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,Unknown,-74.09
3,Payment,00205734,Payment,col0265,NaN,NaN,NaN,2021-12-01,NaT,No Description,...,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,Unknown,-400.04
4,Payment,00205720,Payment,jen0078,NaN,NaN,NaN,2021-12-01,NaT,No Description,...,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,0.0,Unknown,-128.90


## Unit Cost Sell is a disaster.

## There are items with no unit cost sell, items with unit cost sell above unit price sell, and items with unit cost sell = unit price sell, this data is not useable in its current state. 

## I could try to do a reasonable estimation on the unit cost sell, but it would be just that, an estimation. I may do this in the future, but for now I want to work with data that has been validated and known to be correct, not guesses

In [53]:
data.loc[(data['Unit Cost Sell'] >= data['Unit Price Sell'])].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12270 entries, 0 to 48871
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   History Type            12270 non-null  object        
 1   Transaction ID          12270 non-null  object        
 2   Transaction Type        12270 non-null  object        
 3   Customer ID             12270 non-null  object        
 4   Ship To City            6729 non-null   object        
 5   Ship To Region          6631 non-null   object        
 6   Ship To Country         6742 non-null   object        
 7   Transaction Date        12270 non-null  datetime64[ns]
 8   Act Ship Date           6333 non-null   datetime64[ns]
 9   Description             12270 non-null  object        
 10  Additional Description  12270 non-null  object        
 11  Item ID                 12270 non-null  object        
 12  Drop Ship               12270 non-null  object

**removing unit cost sell from data. Can also remove Description and Additional Description as I have gotten the information that I needed from these columns**

In [54]:
del data['Unit Cost Sell']
del data['Description']
del data['Additional Description']

In [55]:
data

,History Type,Transaction ID,Transaction Type,Customer ID,Ship To City,Ship To Region,Ship To Country,Transaction Date,Act Ship Date,Item ID,Drop Ship,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Ship Via,Amount
0,Payment,00205712,Payment,bet0095,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-100.79
1,Payment,00204681,Payment,Tod0111,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-620.62
2,Payment,00205719,Payment,chr0867,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-74.09
3,Payment,00205734,Payment,col0265,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-400.04
4,Payment,00205720,Payment,jen0078,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-128.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48867,Line Item,00174473,Invoice,ada0153,Colorado Springs,CO,USA,2021-01-02,2021-01-11,YPV,N,B01,Hoses & Acce,1.0,EA,39.95,0.0,Amount,GND,39.95
48868,Freight,00174473,Invoice,ada0153,Colorado Springs,CO,USA,2021-01-02,2021-01-11,No Item ID,N,B01,No Product Line,1.0,No Units,26.45,0.0,Amount,GND,26.45
48869,Payment,00191070,Payment,ran0051,NaN,NaN,NaN,2021-01-02,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-1592.50
48870,Payment,00191077,Payment,nic0216,NaN,NaN,NaN,2021-01-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.00,0.0,Amount,Unknown,-1076.35


## Looking at Address data that is NaN

In [56]:
data.isna().sum()

History Type           0
Transaction ID         0
Transaction Type       0
Customer ID            0
Ship To City        5608
Ship To Region      6646
Ship To Country     5528
Transaction Date       0
Act Ship Date       7345
Item ID                0
Drop Ship              0
Location ID         5543
Product Line           0
Qty Ship Sell          0
Units                  0
Unit Price Sell        0
Price Adj Pct          0
Price Adj Type         0
Ship Via               0
Amount                 0
dtype: int64

**So I only need Region information for orders made in the USA (since my visualizations for international orders will be at country level, not region level). So I will need to fill in the Regions in the USA that are empty**

In [57]:
data.loc[((data['Ship To Region'].isna()) & (data['Ship To Country'].isin(['USA', 'UNITED']))), ['Ship To City']].value_counts()

Ship To City
Bloomington     73
Evansville       6
esperance        1
dtype: int64

In [58]:
#Bloomington and Evansville are in the State of Indiana. going to explore esperance

In [59]:
data.loc[(data['Ship To City'] == 'esperance')]

#we are only interested with the City Esperance in the USA. Looking in Traverseon the Transaction ID 00179220, the state is WA

,History Type,Transaction ID,Transaction Type,Customer ID,Ship To City,Ship To Region,Ship To Country,Transaction Date,Act Ship Date,Item ID,Drop Ship,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Ship Via,Amount
16710,Line Item,00179220,Credit Memo,bea0047,esperance,NaN,USA,2021-07-21,2021-07-21,sublined item,N,B01,service,-1.0,ea,145.00,0.0,Percent,Unknown,-145.00
17298,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,ABR INSURANCE,N,B01,INSURANCE,1.0,EA,25.30,0.0,Amount,Request FREIGHT or I,25.30
17299,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,Handling and-or cod fee,N,B01,service,1.0,EA,49.25,0.0,Amount,Request FREIGHT or I,49.25
17300,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,854-214,N,B01,Swivels,1.0,Each,50.00,0.0,Amount,Request FREIGHT or I,50.00
17301,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,GTT-BT-Phantom-4S,N,B01,GTT,1.0,EACH,1200.00,0.0,Amount,Request FREIGHT or I,1200.00
17302,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,bunsenh-5600,N,B01,Bunsen Burne,1.0,Each,68.00,0.0,Amount,Request FREIGHT or I,68.00
17303,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,ggt-Rock-ItFP,N,B01,FOOT PEDALS,1.0,EA,440.00,0.0,Amount,Request FREIGHT or I,440.00
17304,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,cshear8,N,B01,CUTTING TOOL,1.0,EACH,45.00,0.0,Amount,Request FREIGHT or I,45.00
17305,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,2021-07-20,ggt-ct,N,B01,CLAWS,1.0,EA,65.00,0.0,Amount,Request FREIGHT or I,65.00
17306,Line Item,00179135,Invoice,bea0047,esperance,WA,AUS,2021-07-16,NaT,BS-BS-M-Phantom,Y,B01,Torch Marver,0.0,EA,145.00,0.0,Amount,Request FREIGHT or I,0.00


**Adding regions for Bloomington, Evansville, and Esperance**

In [60]:
data.loc[((data['Ship To Region'].isna()) & (data['Ship To City'] == 'Bloomington')), 'Ship To Region'] = 'IN'

data.loc[((data['Ship To Region'].isna()) & (data['Ship To City'] == 'Evansville')), 'Ship To Region'] = 'IN'

data.loc[((data['Ship To Region'].isna()) & (data['Ship To Country'] == 'USA') & (data['Ship To City'] == 'esperance')), 'Ship To Region'] = 'WA'

In [61]:
data[data['Transaction ID'] == '00179220']

#great, this worked

,History Type,Transaction ID,Transaction Type,Customer ID,Ship To City,Ship To Region,Ship To Country,Transaction Date,Act Ship Date,Item ID,Drop Ship,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Ship Via,Amount
16710,Line Item,00179220,Credit Memo,bea0047,esperance,WA,USA,2021-07-21,2021-07-21,sublined item,N,B01,service,-1.0,ea,145.0,0.0,Percent,Unknown,-145.0


In [62]:
#looking through orders that have USA as shipping country but NaN Region.
data[(data['Ship To Country'] == 'USA') & (data['Ship To Region'].isna())]

,History Type,Transaction ID,Transaction Type,Customer ID,Ship To City,Ship To Region,Ship To Country,Transaction Date,Act Ship Date,Item ID,Drop Ship,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Ship Via,Amount
13163,Line Item,00179698,Invoice,suz0056,NaN,NaN,USA,2021-08-19,2021-08-19,Glassseparator,N,B01,Kiln Accesso,1.0,EA,7.784,0.0,Percent,Unknown,7.78
13164,Sales Tax,00179698,Invoice,suz0056,NaN,NaN,USA,2021-08-19,2021-08-19,No Item ID,N,B01,No Product Line,1.0,No Units,0.540,0.0,Amount,Unknown,0.54
14067,Line Item,00179569,Invoice,dav0534,NaN,NaN,USA,2021-08-12,2021-08-12,GLOa1lb,N,B01,GLOW POWDER,1.0,5GR,5.000,0.0,Percent,Pick Up Thursday,5.00
14068,Line Item,00179569,Invoice,dav0534,NaN,NaN,USA,2021-08-12,2021-08-12,DH20HOLETIP,N,B01,National Com,1.0,each,50.000,0.0,Percent,Pick Up Thursday,50.00
14069,Line Item,00179569,Invoice,dav0534,NaN,NaN,USA,2021-08-12,2021-08-12,BB-BoroBatch-Shorts,N,B01,Boro Batch,1.0,LB,15.000,0.0,Percent,Pick Up Thursday,15.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41079,Line Item,00175502,Invoice,rus0049,NaN,NaN,USA,2021-02-12,2021-02-12,glasscuttr,N,B01,CUTTING TOOL,1.0,EACH,15.950,0.0,Percent,Pick Up,15.95
41080,Line Item,00175502,Invoice,rus0049,NaN,NaN,USA,2021-02-12,2021-02-12,246190250,N,B01,Simax Tubing,1.0,PC,6.343,0.0,Percent,Pick Up,6.34
41081,Line Item,00175502,Invoice,rus0049,NaN,NaN,USA,2021-02-12,2021-02-12,246190320,N,B01,Simax Tubing,1.0,PC,7.766,0.0,Percent,Pick Up,7.77
41082,Line Item,00175502,Invoice,rus0049,NaN,NaN,USA,2021-02-12,NaT,854-399,N,B01,CUTTING TOOL,0.0,EA,54.950,0.0,Percent,Pick Up,0.00


In [63]:
#so any order with sales tax here is a pick up order. All the orders above have Sales Tax, so the City and Region can be set to Bloomington, IN
data.loc[(data['Ship To Country'] == 'USA') & (data['Ship To Region'].isna()), 'Ship To City'] = 'Bloomington'

data.loc[(data['Ship To Country'] == 'USA') & (data['Ship To Region'].isna()), 'Ship To Region'] = 'Indiana'

In [64]:
#this is confirming that the remaining NaN regions in the data is in foreign countries and therefore do not need this information
data[data['Ship To Region'].isna()]['Ship To Country'].value_counts()

JPN       283
DEU       217
FRA       137
GBR        80
ITA        41
AUS        35
MEX        32
FRANCE     27
THA        25
CHL        24
ARG        14
NZL        12
BEL        11
IRL        10
JAPAN       7
CHN         4
IND         3
CHE         2
Name: Ship To Country, dtype: int64

In [65]:
#looing at the remaining NaN cities
data[data['Ship To City'].isna()]

,History Type,Transaction ID,Transaction Type,Customer ID,Ship To City,Ship To Region,Ship To Country,Transaction Date,Act Ship Date,Item ID,Drop Ship,Location ID,Product Line,Qty Ship Sell,Units,Unit Price Sell,Price Adj Pct,Price Adj Type,Ship Via,Amount
0,Payment,00205712,Payment,bet0095,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-100.79
1,Payment,00204681,Payment,Tod0111,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-620.62
2,Payment,00205719,Payment,chr0867,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-74.09
3,Payment,00205734,Payment,col0265,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-400.04
4,Payment,00205720,Payment,jen0078,NaN,NaN,NaN,2021-12-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-128.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48702,Payment,00191080,Payment,ada0153,NaN,NaN,NaN,2021-01-02,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-265.24
48703,Payment,00191078,Payment,die0005,NaN,NaN,NaN,2021-01-02,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-798.49
48869,Payment,00191070,Payment,ran0051,NaN,NaN,NaN,2021-01-02,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-1592.50
48870,Payment,00191077,Payment,nic0216,NaN,NaN,NaN,2021-01-01,NaT,No Item ID,N,NaN,No Product Line,0.0,No Units,0.0,0.0,Amount,Unknown,-1076.35


In [66]:
# for Transaction ID 0018138, looked in Traverse and the city is Atlanta, will change this
data.loc[(data['Transaction ID'] == '00181389'), 'Ship To City'] = 'Atlanta'

**removing History type of Payment**

In [67]:
data = data.loc[(data['History Type'] != 'Payment')]

## Great! I believe this data is ready for Visualization. will write to a file

In [68]:
data.to_csv('ABR Imagery 2021 Orders.csv', index = False)